In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, END, START
from graph_state import AgentState, parser as supervisor_parser
from prompt_supervisor import get_supervisor_system_prompt
import os

In [ ]:
stp = get_supervisor_system_prompt()
user_query = "Can you explain the key differences between post-trade processing for securities and commodities, including their settlement mechanisms?"

In [ ]:
outparser = stp.invoke({'question': user_query})

In [ ]:
print(outparser)

In [ ]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage

In [ ]:
model = ChatGoogleGenerativeAI(model='gemini-1.5-flash')

In [ ]:
## Workflow
workflow = StateGraph(AgentState)

In [ ]:
## supervisor function
def supervisor_func(state: AgentState):
    print("->Supervisor node call->")
    messages = state.get('messages', [])
    question = ''
    if (len(messages) > 0):
        question = messages[0]

    if question:
        prompt = get_supervisor_system_prompt()

        chain = prompt | model | supervisor_parser

        output = chain.invoke({'question': question})

        parsed_output = output.model_dump()

        print('##supervisor output', parsed_output)

    return {'messages': [f"{parsed_output["node_selection_type"]}"]}

In [ ]:
## llm function
def llm_func(state):
    print("->LLM node call->")
    return state

In [ ]:
## RAG function
def rag_func(state):
    print("->RAG node call->")
    return state

In [ ]:
## web_crawler function
def web_crawler_func(state):
    print("->Crawler node call->")
    return state

In [ ]:
## Router function
def router(state: AgentState):
    print("->Router node call->")
    messages = state.get('messages', [])
    latest_message = ''
    if (len(messages) > 0):
        latest_message = messages[-1]
    return 'RAG Call' if 'rag' in latest_message else 'LLM Call' if 'llm' in latest_message else 'Crawler Call'


In [ ]:
## Validation function
def validation_func(state):
    print("->Validation node call->")
    return state

In [ ]:
## Validation router
def validation_router(state):
    print("->Validation router node call->")
    return 'END Call'

In [ ]:
workflow.add_node('supervisor', supervisor_func)
workflow.add_node('llm', llm_func)
workflow.add_node('crawler', web_crawler_func)
workflow.add_node('rag', rag_func)
workflow.add_node('validation', validation_func)

In [ ]:
workflow.add_edge(START, 'supervisor')
workflow.add_conditional_edges(
    'supervisor',
    router,
    {
        'RAG Call': 'rag',
        'LLM Call': 'llm',
        'Crawler Call': 'crawler'
    }
)
workflow.add_edge('rag', 'validation')
workflow.add_edge('llm', 'validation')
workflow.add_edge('crawler', 'validation')

In [ ]:
workflow.add_conditional_edges(
    'validation',
    validation_router,
    {
        'Supervisor Call': 'supervisor',
        'END Call': END
    }
)

In [ ]:
app = workflow.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
user_query_rag = "What are the key steps involved in the post-trade settlement process for commodities, and how does it differ from securities?"
user_query_crawler = "Find the latest commodity price trends for crude oil and gold."
user_query_llm = "Give me a creative summary of how commodities trading has evolved over the past decade."
app_state:AgentState = {'messages': [user_query_rag]} 

In [ ]:
for output in app.stream(app_state):
    print('##output', output)